# Distributed Collective Communication

> FlashAttention optimizes movement within one GPU; distributed models must also exchange results across devices. If eight GPUs compute separate gradients, every GPU needs the same average to keep parameters synchronized. `all_reduce` combines aggregation and distribution.
>
> **Participants**: Rank identifies a process, World Size is the participant count, and a Process Group defines the devices in one communication operation.
>
> **Semantics and algorithms**: broadcast, reduce, all-reduce, all-gather, reduce-scatter, and all-to-all specify data owners, result owners, and aggregation. Ring balances transfers; all-to-all rearranges data for MoE. Identical semantics can have very different costs under different algorithms and topologies.

We begin with a centralized bottleneck, define the participants, and compare algorithms that distribute communication more evenly.


## 0. The Basic Collective-Communication Problem

Each GPU computes locally, but some results must become globally consistent. In gradient synchronization, divergent gradients would make model replicas diverge after the optimizer step.

A naive design sends all gradients to rank 0, averages them, and broadcasts the result. It is correct, but rank 0 receives and sends $N-1$ tensors and becomes the bottleneck. Collective algorithms aim to perform collection, combination, and distribution quickly and fairly across all devices.


### Three Basic Terms

- **Rank**: process or device index, starting from 0.
- **World Size**: total number of participating processes.
- **Process Group**: the subset participating in a particular operation.

Together they answer “who am I?”, “how many participants exist?”, and “who joins this operation?”


## 1. Point-to-Point Communication

Collectives are built from point-to-point `send` and `recv`. Blocking calls wait until the peer participates. Incorrect ordering can deadlock—for example, two processes both blocking on send before either receives. Frameworks commonly use nonblocking primitives internally, while training code normally calls higher-level collectives.


## 2. Seven Standard Collective Operations

Their essential differences are who owns input, who receives output, and whether values are aggregated.

| Operation | Meaning | Input → Output | Typical Use |
|:---|:---|:---|:---|
| broadcast | copy one value to everyone | 1 → all, identical | initialize parameters |
| scatter | distribute distinct slices | 1 → all, sliced | distribute a batch |
| gather | concatenate slices at one rank | all → 1 | collect results |
| reduce | aggregate at one rank | all → 1, reduced | aggregate loss |
| all-reduce | aggregate for everyone | all → all, reduced | DDP gradients |
| all-gather | concatenate for everyone | all → all, concatenated | FSDP parameters |
| reduce-scatter | aggregate then partition | all → all, reduced slices | ZeRO gradients |


### 2.1 Broadcast

Rank 0 owns one tensor and copies it to all ranks: one input becomes $N$ identical outputs. It commonly synchronizes initial parameters. Rank 0 sends $N-1$ copies, for $(N-1)D$ total source traffic. Everyone receives the same data.


In [ ]:
# === NumPy simulation of broadcast ===
import numpy as np

# rank 0 owns the original data
np.random.seed(0)
data_on_rank0 = np.array([10, 20, 30, 40])

# Before broadcast, only rank 0 has data; other GPUs are empty
cards_before = [None, None, None, None]
cards_before[0] = data_on_rank0.copy()
print("Before broadcast:")
for rank, data in enumerate(cards_before):
    print(f"  rank {rank}: {data}")

# After broadcast, every GPU has an identical copy
cards_after = [data_on_rank0.copy() for _ in range(4)]
print("\nAfter broadcast:")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\nKey observation: one input becomes N identical outputs.")
print("Traffic = data size × (N-1), because rank 0 sends to the other N-1 GPUs.")


### 2.2 Scatter

Rank 0 partitions a tensor into $N$ slices and sends slice $i$ to rank $i$. Unlike broadcast, outputs differ. It can distribute a large batch for Data Parallelism. Source traffic is $(N-1)D/N$ when the complete input has size $D$.


In [ ]:
# === NumPy simulation of scatter ===
import numpy as np

# rank 0 owns one large array with four rows, ready to split four ways
big_array = np.array([[1, 2],
                      [3, 4],
                      [5, 6],
                      [7, 8]])

print("Before scatter:")
print(f"  rank 0, complete:\n{big_array}")
for rank in range(1, 4):
    print(f"  rank {rank}: empty")

# Scatter along axis 0 into four shards, sending shard i to GPU i
shards = np.split(big_array, 4, axis=0)
print("\nAfter scatter:")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard.ravel()}")

print("\nKey observation: one large array becomes N distinct slices, one per rank.")
print("Unlike broadcast, where everyone gets the same data, scatter gives each rank a different shard.")


### 2.3 Gather

Every rank owns one slice; rank 0 concatenates them in order. It reverses scatter, and only rank 0 receives the complete result. Distributed inference may gather generated sequences for post-processing. Rank 0 receives $(N-1)D/N$.


In [ ]:
# === NumPy simulation of gather ===
import numpy as np

# Every GPU owns one shard
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("Before gather:")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# Gather: rank 0 collects and concatenates all shards in order
gathered_on_rank0 = np.concatenate(shards, axis=0)
print("\nAfter gather:")
print(f"  rank 0, concatenated: {gathered_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: still owns original data {shards[rank]}, not concatenated")

print("\nKey observation: N inputs become one concatenated output on rank 0 only.")


### 2.4 Reduce

Every rank owns a tensor; an element-wise operation such as sum aggregates them at rank 0 only. It differs from gather by combining rather than concatenating values. Use it when only one rank needs the total, such as logging a global loss.


In [ ]:
# === NumPy simulation of reduce ===
import numpy as np

# Every GPU owns one tensor
cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("Before reduce:")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# Reduce-sum adds corresponding positions and leaves the result only at rank 0
result_on_rank0 = np.sum(np.stack(cards), axis=0)
print("\nAfter reduce-sum:")
print(f"  rank 0: {result_on_rank0}")
for rank in range(1, 4):
    print(f"  rank {rank}: no retained result")

print("\nKey observation: N inputs become one reduced output on rank 0 only.")
print("Sum is the most common reduction, though max and min are also available.")


### 2.5 All-Reduce

Every rank contributes a tensor and every rank receives the same aggregate. It is semantically a reduce followed by broadcast. DDP uses it to synchronize average gradients so all replicas remain identical. The next example first implements the naive centralized semantics before introducing Ring All-Reduce.


In [ ]:
# === NumPy simulation of all-reduce; semantics first, algorithm next ===
import numpy as np

cards = [np.array([1.0, 2.0]),
         np.array([3.0, 4.0]),
         np.array([5.0, 6.0]),
         np.array([7.0, 8.0])]

print("Before all-reduce:")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# All-reduce-sum gives the complete sum to every GPU
total = np.sum(np.stack(cards), axis=0)
cards_after = [total.copy() for _ in range(4)]

print("\nAfter all-reduce-sum:")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\nKey observation: N different inputs become N identical reduced outputs.")
print("Unlike reduce, every rank receives the result rather than rank 0 alone.")


### 2.6 All-Gather

Each rank owns a slice; every rank receives the complete concatenation. Gather produces the full tensor only at rank 0, while all-gather produces it everywhere. FSDP uses all-gather before a layer to reconstruct sharded parameters. Each rank receives $(N-1)D/N$.


In [ ]:
# === NumPy simulation of all-gather ===
import numpy as np

# Every GPU owns one shard
shards = [np.array([10, 20]),
          np.array([30, 40]),
          np.array([50, 60]),
          np.array([70, 80])]

print("Before all-gather:")
for rank, s in enumerate(shards):
    print(f"  rank {rank}: {s}")

# All-gather gives every GPU the concatenation of all shards
full = np.concatenate(shards, axis=0)
cards_after = [full.copy() for _ in range(4)]

print("\nAfter all-gather:")
for rank, data in enumerate(cards_after):
    print(f"  rank {rank}: {data}")

print("\nUnlike gather, every rank receives the complete result rather than rank 0 alone.")


### 2.7 Reduce-Scatter

Each rank starts with a full tensor. Values are reduced, then the aggregate is partitioned so each rank keeps one slice. A key identity is

`reduce-scatter + all-gather = all-reduce`.

ZeRO uses it so each rank retains only its assigned gradient shard. Per-rank traffic is $(N-1)D/N$ per phase.


In [ ]:
# === NumPy simulation of reduce-scatter ===
import numpy as np

N = 4
# Every GPU owns one full vector of length N
cards = [np.array([1.0, 2.0, 3.0, 4.0]),
         np.array([10., 20., 30., 40.]),
         np.array([100., 200., 300., 400.]),
         np.array([0.1, 0.2, 0.3, 0.4])]

print("Before reduce-scatter:")
for rank, data in enumerate(cards):
    print(f"  rank {rank}: {data}")

# First sum across GPUs, then split into N shards
stacked = np.stack(cards)          # shape (N,N)
summed = stacked.sum(axis=0)       # each position is summed over GPUs
shards = np.split(summed, N)       # split into N pieces

print("\nAfter summing all GPUs:", summed)
print("\nAfter reduce-scatter:")
for rank, shard in enumerate(shards):
    print(f"  rank {rank}: {shard}")

print("\nKey observation: N full inputs become N reduced slices, leaving one Nth on each rank.")


### 2.8 Comparing the Seven Operations

The following table places direction, volume, and common uses side by side.


In [ ]:
# === Comparison table for seven collectives ===
rows = [
    ("Operator",      "Direction",                 "Traffic",          "Typical use"),
    ("broadcast",     "1 -> all, identical",       "(N-1) × D",       "broadcast parameters at initialization"),
    ("scatter",       "1 -> all, shards",          "(N-1) × D/N",     "rank 0 distributes a batch"),
    ("gather",        "all → 1",                 "(N-1) × D/N",     "rank 0 collects results"),
    ("reduce",        "all -> 1, aggregate",       "(N-1) × D/N",     "rank 0 aggregates loss"),
    ("all-reduce",    "all -> all, aggregate",     "2(N-1)/N × D",    "DDP gradient synchronization"),
    ("all-gather",    "all -> all, concatenate",   "(N-1)/N × D",     "FSDP parameter gathering"),
    ("reduce-scatter", "all -> all, reduce+shard", "(N-1)/N × D",     "ZeRO gradient sharding"),
]

widths = [16, 24, 16, 24]
for row in rows:
    line = " | ".join(f"{c:<{w}}" for c, w in zip(row, widths))
    print(line)
    if row[0] == "Operator":
        print("-" * len(line))

print("\nD is per-GPU data size; N is GPU count.")
print("The (N-1)/N and 2(N-1)/N factors come from the ring algorithm in the next section.")
print("For now, remember that all-to-all is central to MoE; we reserve its slot here.")


## 3. Ring All-Reduce

Centralized aggregation overloads rank 0. Ring All-Reduce arranges ranks in a ring and communicates only with neighbors.

1. **Reduce-scatter**: split each tensor into $N$ chunks. Across $N-1$ rounds, send a chunk rightward and add the chunk arriving from the left. Each rank ends with one globally reduced chunk.
2. **All-gather**: circulate those completed chunks for another $N-1$ rounds until every rank owns the full aggregate.

Traffic is balanced because every rank performs the same transfers. The next example traces four ranks step by step.


### 3.1 Hand-Calculating Four-GPU Ring All-Reduce

Each of four ranks owns a four-element vector and needs the element-wise global sum. Treat every element as one chunk. Three reduce-scatter rounds produce one complete sum chunk per rank; three all-gather rounds circulate those chunks. The code prints every step so you can verify which chunk rank 0 sends to rank 1 in the first round.


In [ ]:
# === Step-by-step ring all-reduce with N=4 ===
import numpy as np

N = 4
np.random.seed(42)
# Every GPU owns a length-four vector split into four one-element shards
cards = [list(np.random.randint(1, 10, size=N)) for _ in range(N)]

print("Initially, every GPU owns its own vector")
for r in range(N):
    print(f"  rank {r}: {cards[r]}")

expected = [sum(cards[r][k] for r in range(N)) for k in range(N)]
print(f"\nExpected result, position-wise sum across GPUs: {expected}")

# ======== Phase 1: reduce-scatter, three rounds sending one shard right and accumulating ========
buffer = [row[:] for row in cards]
print("\n--- Phase 1 reduce-scatter: send one shard right each round and accumulate ---")
for step in range(N - 1):
    # Record the shard each GPU sends this round
    outgoing = {}
    for r in range(N):
        send_idx = (r - step) % N
        outgoing[r] = (send_idx, buffer[r][send_idx])
    # Execute together: each right neighbor adds the received value at that position
    for r in range(N):
        send_idx, value = outgoing[r]
        right = (r + 1) % N
        buffer[right][send_idx] += value
    print(f"After round {step+1}:")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

# Verify that rank r now owns global-sum shard (r+1)%N
print("\nAfter stage one, each rank holds one shard of the global reduced sum")
for r in range(N):
    master_idx = (r + 1) % N
    got = buffer[r][master_idx]
    assert got == expected[master_idx], f"rank {r} shard {master_idx} = {got}"
    print(f"  rank {r} owns shard {master_idx} = {got} ✓")

# ======== Phase 2: all-gather, three rounds distributing sum shards to everyone ========
# Each GPU initially sends the global-sum shard it owns
out_chunk_idx = [(r + 1) % N for r in range(N)]
out_value = [buffer[r][(r + 1) % N] for r in range(N)]

print("\n--- Phase 2 all-gather: circulate each GPU's sum shard around the ring ---")
for step in range(N - 1):
    snapshot_idx = out_chunk_idx[:]
    snapshot_val = out_value[:]
    for r in range(N):
        right = (r + 1) % N
        received_idx = snapshot_idx[r]
        received_val = snapshot_val[r]
        buffer[right][received_idx] = received_val
        # In the next round, the right neighbor forwards the shard it just received
        out_chunk_idx[right] = received_idx
        out_value[right] = received_val
    print(f"After round {step+1}:")
    for r in range(N):
        print(f"    rank {r}: {buffer[r]}")

# Verification
all_ok = all(buffer[r] == expected for r in range(N))
print(f"\nFinal result: does every GPU hold the global sum? {all_ok}")
for r in range(N):
    print(f"  rank {r}: {buffer[r]}")

print("\nKey observation:")
print(f"  Each phase takes {N-1} rounds, and every GPU sends only one element per round.")
print(f"  Total traffic per GPU = 2 × (N-1) elements, nearly independent of N.")
print(f"  A centralized rank 0 would receive four and send four, totaling eight and becoming the bottleneck.")


### 3.2 Communication Efficiency of Ring All-Reduce

| Method | Per-Rank Communication | Load |
|:---|:---|:---|
| centralized | rank 0 transfers $2(N-1)D$ | rank 0 bottleneck |
| ring | every rank transfers $2(N-1)D/N$ | balanced |

As $N$ grows, Ring traffic per rank approaches $2D$ instead of growing with $N$. The same “split into chunks and circulate” principle appears in efficient all-gather and reduce-scatter implementations.


## 4. All-to-All

Every rank holds $N$ pieces, one intended for each destination. After all-to-all, rank $i$ has received its designated piece from every source rank. This is a global rearrangement rather than aggregation and cannot use the same Ring reduction pattern.

MoE relies on all-to-all because Tokens must travel to the GPUs that own their selected experts and return afterward.


In [ ]:
# === NumPy simulation of all-to-all ===
import numpy as np

N = 4
# Each GPU send buffer: shard i is destined for rank i
# Each element is (sender, receiver), making the direction explicit
send_buffers = [
    np.array([[0, 0], [0, 1], [0, 2], [0, 3]]),  # rank 0 shards for ranks 0,1,2,3
    np.array([[1, 0], [1, 1], [1, 2], [1, 3]]),  # rank 1 shards
    np.array([[2, 0], [2, 1], [2, 2], [2, 3]]),  # rank 2 shards
    np.array([[3, 0], [3, 1], [3, 2], [3, 3]]),  # rank 3 shards
]

print("Before all-to-all: each GPU's send array, with row i targeting rank i")
for rank, buf in enumerate(send_buffers):
    print(f"  rank {rank}:")
    print(buf)

# All-to-all transposes the two-dimensional sender-receiver table
all_data = np.transpose(np.stack(send_buffers), (1, 0, 2))

print("\nAfter all-to-all: each GPU's received array, with row j coming from rank j")
for rank in range(N):
    print(f"  rank {rank}:")
    print(all_data[rank])

print("\nKey observation: all ranks exchange data simultaneously; each sends N pieces and receives N pieces.")
print("Communication volume is N times the data size; ring reduction cannot distribute this cost, making it expensive.")


## 5. Notation for Sharded Layouts

Let matrix axes be $I$ (rows) and $J$ (columns), and device-grid axes be $X$ and $Y$.

- `A[I_X, J]`: rows are partitioned along device axis X; columns are replicated.
- `A[I, J_Y]`: columns are partitioned along device axis Y; rows are replicated.

Rule: an axis appearing as a subscript is sharded; an unsubscripted axis is replicated. The next example uses a 2×2 device grid.


In [ ]:
# === partition notation：A[I_X, J] vs A[I, J_Y] ===
import numpy as np

# Full 4×4 matrix A distributed over a 2×2 grid of four GPUs
A = np.arange(16).reshape(4, 4)
print("Full matrix A, shape (4,4):")
print(A)
print()

print("=== A[I_X, J]: rows sharded over X, columns replicated ===")
print("X has two positions, so rows split into two shards of two rows each")
row_shards = np.split(A, 2, axis=0)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):")
        print(row_shards[x])

print("=== A[I, J_Y]: columns sharded over Y, rows replicated ===")
col_shards = np.split(A, 2, axis=1)
for x in range(2):
    for y in range(2):
        print(f"  device (x={x}, y={y}):")
        print(col_shards[y])

print("\nKey observation: an index present in the output means sharding; an absent index means replication.")
print("The same data under different partitions gives every GPU a different submatrix.")


### 5.1 Sharding Notation for Collectives

- **all-gather**: `A[I, J_Y] → A[I, J]`, removing a column shard.
- **reduce-scatter**: `A[I, J] → A[I, J_Y]`, reducing then adding a shard.
- **all-reduce**: reduce-scatter followed by all-gather; the layout stays replicated while values change.

All-gather removes partitioning; reduce-scatter introduces partitioning after aggregation.


## 6. Collective Communication with `torch.distributed`

PyTorch `torch.distributed` wraps NCCL on GPUs and can use Gloo on CPUs. The workflow is:

1. call `dist.init_process_group(...)` with rank and World Size;
2. call a collective such as `dist.all_reduce(tensor)`, which modifies the tensor in place;
3. call `dist.destroy_process_group()`.

The example launches two Gloo processes. Multiprocessing may be awkward inside Jupyter, so understanding the logic is sufficient if the environment cannot run it.


In [ ]:
# === Real two-process torch.distributed all-reduce example ===
# Start two local processes with the gloo backend, using CPU communication and no GPU
# multiprocessing can be awkward in Jupyter; understanding the logic is sufficient

import os
import torch
import torch.multiprocessing as mp
import torch.distributed as dist


def worker_fn(rank, world_size):
    """Each subprocess initializes, runs all-reduce, then cleans up."""
    # 1. Initialize process group; env:// reads the address from environment variables
    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group(backend="gloo", rank=rank, world_size=world_size)

    # 2. Each GPU owns a tensor: rank 0 [1,2], rank 1 [2,3]
    tensor = torch.tensor([float(rank + 1), float(rank + 2)])
    print(f"[before] rank {rank}: {tensor.tolist()}")

    # 3. All-reduce sum is in-place and directly modifies tensor
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    print(f"[after]  rank {rank}: {tensor.tolist()}  (expected [3.0, 5.0])")

    # 4. Clean up
    dist.destroy_process_group()


# Start two processes
if __name__ == "__main__":
    world_size = 2
    mp.start_processes(worker_fn, args=(world_size,), nprocs=world_size,
                       join=True, start_method="fork")
    print("\nBoth ranks obtain the all-reduce result [3.0, 5.0].")


In [ ]:
# === Quick reference for common torch.distributed APIs ===
api_reference = """
# Initialize once when each process starts
dist.init_process_group(backend="nccl" | "gloo", rank=0, world_size=8)

# all-reduce: modify tensor in place; everyone receives the aggregate
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)

# broadcast: send from src to everyone
dist.broadcast(tensor, src=0)

# all-gather: output is a list of length world_size
out_list = [torch.empty_like(tensor) for _ in range(world_size)]
dist.all_gather(out_list, tensor)

# reduce-scatter: input is a list; output is one tensor
in_list = [torch.rand(4) for _ in range(world_size)]
out = torch.empty(4)
dist.reduce_scatter(out, in_list, op=dist.ReduceOp.SUM)

# all-to-all, single-tensor version commonly used for MoE routing
out = torch.empty_like(input_tensor)
dist.all_to_all_single(out, input_tensor)
"""
print(api_reference)
print("Memory aids:")
print("  - all_reduce and broadcast are in-place and directly modify tensor")
print("  - all_gather and reduce_scatter transfer through lists")
print("  - aggregation defaults to SUM")


## Summary

- [ ] Rank, World Size, and Process Group define participants.
- [ ] Broadcast copies one value; scatter distributes distinct slices.
- [ ] Gather concatenates at one rank; reduce aggregates at one rank.
- [ ] All-reduce aggregates for everyone and synchronizes DDP gradients.
- [ ] All-gather reconstructs shards for everyone; reduce-scatter aggregates and retains shards.
- [ ] Reduce-scatter followed by all-gather equals all-reduce.
- [ ] Ring All-Reduce balances traffic at about twice the tensor size per rank.
- [ ] All-to-all rearranges data and is central to MoE.
- [ ] In layout notation, a subscripted device axis indicates sharding.


## Exercises

> You may ask AI for hints or direction checks, but avoid asking it to complete the exercise.

**Exercise 1: Ring All-Reduce Communication Volume**

An eight-GPU Ring All-Reduce sums a 64 MB tensor. How many MB does each GPU send and receive in total? In a centralized design, how much traffic does rank 0 handle?

Hint: each of two Ring phases has $N-1$ rounds transferring $D/N$; centralized rank 0 receives and sends $(N-1)D$.


In [ ]:
# Exercise 1: ring all-reduce versus a centralized scheme
D_mb = 64    # 64 MB tensor
N = 8        # eight GPUs

# TODO: per-GPU traffic in MB
ring_per_card = None

# TODO: traffic through rank 0 in the centralized scheme
naive_rank0 = None

assert ring_per_card is not None and naive_rank0 is not None, 'Please replace the placeholder before running the assertion.'
# Ring = reduce-scatter + all-gather; each phase has (N-1) rounds × D/N
expected_ring = 2 * (N - 1) * D_mb / N
# Centralized: rank 0 receives (N-1)*D, then sends (N-1)*D
assert abs(ring_per_card - expected_ring) < 0.1, f"Ring should be {expected_ring:.1f} MB"
assert abs(naive_rank0 - 2 * (N - 1) * D_mb) < 0.1, \
    f"Centralized rank 0 should handle {2 * (N - 1) * D_mb} MB"

print(f"Exercise 1 passed:")
print(f"  Ring: {ring_per_card:.1f} MB per GPU, evenly balanced")
print(f"  Centralized: rank 0 alone handles {naive_rank0} MB and becomes the bottleneck")
print(f"  As GPU count grows, centralized traffic rises linearly while ring barely changes.")


**Exercise 2: Describe an FSDP Forward Pass with Notation**

FSDP stores `W[I, J_X]` across $N$ GPUs and all-gathers it before use. Fill in the resulting layout.

Hint: all-gather removes the shard annotation.


In [ ]:
# Exercise 2: partition notation for FSDP forward all-gather

# TODO: form of W before forward, sharded along X
fsdp_input = None

# TODO: form of W after all-gather, no longer sharded
fsdp_output = None

assert fsdp_input == "W[I, J_X]", "Before FSDP forward, parameters are sharded along X as W[I, J_X]"
assert fsdp_output == "W[I, J]", "All-gather removes the partition, giving W[I, J]"

print("Exercise 2 passed!")
print(f"  Before forward: {fsdp_input}, each GPU stores only 1/N of parameters")
print(f"  After all-gather: {fsdp_output}, each GPU temporarily owns all parameters")
print(f"  During backward, reduce-scatter shards them again, matching the notation from Section 5.1.")


**Exercise 3: MoE All-to-All Volume**

For batch 512, sequence 2048, hidden size 4096, top-k 2, and BF16, compute per-GPU traffic for EP=4 and EP=8 in one MoE layer.

Hint: one all-to-all sends $(batch	imes seq/EP)	imes top_k	imes hidden	imes2$ bytes per GPU; a layer dispatches and returns.


In [ ]:
# Exercise 3: all-to-all traffic at different EP degrees
batch = 512
seq = 2048
hidden = 4096
top_k = 2
bytes_elem = 2   # BF16


def ep_all_to_all_gb(n_ep):
    """Per-GPU total all-to-all traffic in one MoE layer, in GB."""
    # TODO: implement here
    return None


comm_4 = ep_all_to_all_gb(4)
comm_8 = ep_all_to_all_gb(8)

assert comm_4 is not None and comm_8 is not None, 'Please replace the placeholder before running the assertion.'


def expected(n_ep):
    tokens_per_card = batch * seq / n_ep
    bytes_per_send = tokens_per_card * top_k * hidden * bytes_elem
    return 2 * bytes_per_send / 1e9   # two all-to-alls per layer


assert abs(comm_4 - expected(4)) < 0.01, f"EP=4 should be {expected(4):.2f} GB"
assert abs(comm_8 - expected(8)) < 0.01, f"EP=8 should be {expected(8):.2f} GB"

print(f"Exercise 3 passed:")
print(f"  EP=4: per-GPU all-to-all in one layer {comm_4:.2f} GB")
print(f"  EP=8: per-GPU all-to-all in one layer {comm_8:.2f} GB")
print(f"  Doubling EP halves per-GPU traffic to {comm_8/comm_4*100:.0f}%,")
print(f"  but all-to-all peer count rises with GPUs, so larger EP is not always better.")


## References

- [NCCL Documentation](https://docs.nvidia.com/deeplearning/nccl/) — NVIDIA GPU collectives
- [PyTorch `torch.distributed`](https://pytorch.org/docs/stable/distributed.html) — official API
- Horovod — Ring All-Reduce engineering
- ZeRO — reduce-scatter and all-gather
- Megatron-LM — Tensor Parallelism and layout notation
- DeepSeek-MoE — all-to-all for experts
